Run following command in terminal:

sbatch --gpus=2 --gres=gpumem:40g --time=05:00:00 --mem-per-cpu=32g --wrap="jupyter nbconvert --to notebook --execute 01_251029_generating_description_Apertus-8B-Instruct-2509s.ipynb --inplace"

In [4]:
import pandas as pd

initial_groups_df = pd.read_excel('./data/251027 input data points and groups.xlsx')
initial_groups_df['Data_groups'] = initial_groups_df['Data_groups']\
    .str.strip().str.lower().str.replace('&', 'and')
initial_groups_df.head()

,Data_groups,Data_points,Source,File,Passport_type
0,general information,"Product commercial name, Manufacturer's name, ...",Munaro and Tavares 2020,Munaro and Tavares 2021.pdf,Material passport
1,material health,"Security information, warnings, recommendation...",Munaro and Tavares 2020,Munaro and Tavares 2021.pdf,Material passport
2,sustainability,"Environmental declaration, Life cycle assessme...",Munaro and Tavares 2020,Munaro and Tavares 2021.pdf,Material passport
3,design and production,"Manufacturing process, Manufacturing technique...",Munaro and Tavares 2020,Munaro and Tavares 2021.pdf,Material passport
4,use and operate phase,"Positioning in the building, location in the b...",Munaro and Tavares 2020,Munaro and Tavares 2021.pdf,Material passport


In [5]:
initial_points_df = initial_groups_df.assign(Data_points=initial_groups_df['Data_points']\
    .str.replace('&', 'and').str.split(',')).explode('Data_points').reset_index()
initial_points_df['Data_points'] = initial_points_df['Data_points']\
    .str.strip().str.lower()

initial_points_df = initial_points_df[initial_points_df["Data_points"] != ""].reset_index(drop=True)

initial_points_df.head()

,index,Data_groups,Data_points,Source,File,Passport_type
0,0,general information,product commercial name,Munaro and Tavares 2020,Munaro and Tavares 2021.pdf,Material passport
1,0,general information,manufacturer's name,Munaro and Tavares 2020,Munaro and Tavares 2021.pdf,Material passport
2,0,general information,manufacturer's details,Munaro and Tavares 2020,Munaro and Tavares 2021.pdf,Material passport
3,0,general information,materials composition,Munaro and Tavares 2020,Munaro and Tavares 2021.pdf,Material passport
4,0,general information,product properties,Munaro and Tavares 2020,Munaro and Tavares 2021.pdf,Material passport


In [6]:
initial_points_df.shape

(2010, 6)

In [8]:
initial_points_df.groupby('Passport_type').apply(lambda x: x.shape[0])

/scratch/tmp.52063129.svangelova/ipykernel_191523/2579562470.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  initial_points_df.groupby('Passport_type').apply(lambda x: x.shape[0])


Passport_type
Digital passport                  185
Digital product passport          818
Material passport                 849
Product Circularity Data Sheet    158
dtype: int64

In [4]:
import os

count = 0

files = [f for f in os.listdir("../data/literature/") if os.path.isfile(os.path.join("../data/literature/", f))]

for file in files:
    if file not in initial_points_df.File.unique():
        print (file)
len(files)

30

In [5]:
for source in initial_points_df.File.unique():
    if os.path.isfile(f"../data/literature/{source}"):
        count += 1
    else:
        print("Missing source file!")
count, initial_points_df.File.unique().shape

(30, (30,))

In [6]:
initial_points_df['description'] = ""
initial_points_df.tail()

,index,Data_groups,Data_points,Source,File,Passport_type,description
2005,324,essential environmental characteristics,eco-toxicity,CPR 2024,CPR 2024.pdf,Digital product passport,
2006,324,essential environmental characteristics,freshwater,CPR 2024,CPR 2024.pdf,Digital product passport,
2007,324,essential environmental characteristics,human toxicity cancerogenic,CPR 2024,CPR 2024.pdf,Digital product passport,
2008,324,essential environmental characteristics,human toxicity non-cancerogenic,CPR 2024,CPR 2024.pdf,Digital product passport,
2009,324,essential environmental characteristics,land use related impacts,CPR 2024,CPR 2024.pdf,Digital product passport,


In [10]:
from collections import Counter
Counter(initial_points_df.Passport_type)

Counter({'Material passport': 849,
         'Digital product passport': 818,
         'Digital passport': 185,
         'Product Circularity Data Sheet': 158})

In [11]:
from llama_index.core import Settings, VectorStoreIndex, SimpleDirectoryReader
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface import HuggingFaceLLM
from transformers import AutoTokenizer
from llama_index.core import PromptTemplate

In [9]:
stopping_ids = {
  "<|endoftext|>": 32000,
  "<|assistant|>": 32001,
  "<|placeholder1|>": 32002,
  "<|placeholder2|>": 32003,
  "<|placeholder3|>": 32004,
  "<|placeholder4|>": 32005,
  "<|system|>": 32006,
  "<|end|>": 32007,
  "<|placeholder5|>": 32008,
  "<|placeholder6|>": 32009,
  "<|user|>": 32010
}

In [12]:
import logging
import sys
import torch

logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger().addHandler(logging.StreamHandler(stream=sys.stdout))

In [13]:
from llama_index.core import PromptTemplate
from transformers import BitsAndBytesConfig

system_prompt = """<|system|>
You are a helpful assistant with expertise in sustainable construction.

Rules:
1. Use exactly three sentences. 
2. Base your answer on the provided text.
2. Do not add extra explanations, commentary, or unrelated information.<|end|>
"""

# This will wrap the default prompts that are internal to llama-index
query_wrapper_prompt = PromptTemplate("<|USER|>\n{query_str}\n<|ASSISTANT|>")

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)


llm = HuggingFaceLLM(
    context_window=4096,
    max_new_tokens=500,
    generate_kwargs={"temperature": 0.1, "do_sample": True},
    system_prompt=system_prompt,
    query_wrapper_prompt=query_wrapper_prompt,
    tokenizer_name="microsoft/Phi-4-mini-instruct",
    model_name="microsoft/Phi-4-mini-instruct",
    device_map="auto",
    stopping_ids=[32000, 32001, 32007],
    tokenizer_kwargs={"max_length": 4096},
    model_kwargs={"quantization_config": quantization_config}, # Pass in quantization config
)

Settings.llm = llm
Settings.chunk_size = 1024

INFO:accelerate.utils.modeling:We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).
We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [14]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# TODO: load the embedding model in llama-index setting
Settings.embed_model = HuggingFaceEmbedding(
    model_name="../models/BAAI-bge-small-en-v1.5"
)


INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: ../models/BAAI-bge-small-en-v1.5
Load pretrained SentenceTransformer: ../models/BAAI-bge-small-en-v1.5


In [15]:
file_dir = "../data/literature"
persist_dir = "../storage/lyft"

# For testing only
# Settings.llm = None

documents = SimpleDirectoryReader(file_dir).load_data()
index = VectorStoreIndex.from_documents(
    documents,
)

index.storage_context.persist(persist_dir="../storage/lyft")

query_engine = index.as_query_engine(response_mode="compact")
    
    

Ignoring wrong pointing object 7 0 (offset 0)
Ignoring wrong pointing object 9 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 22 0 (offset 0)
Ignoring wrong pointing object 24 0 (offset 0)
Ignoring wrong pointing object 26 0 (offset 0)
Ignoring wrong pointing object 28 0 (offset 0)
Ignoring wrong pointing object 30 0 (offset 0)
Ignoring wrong pointing object 38 0 (offset 0)
Ignoring wrong pointing object 43 0 (offset 0)
Ignoring wrong pointing object 45 0 (offset 0)
Ignoring wrong pointing object 47 0 (offset 0)
Ignoring wrong pointing object 50 0 (offset 0)
Ignoring wrong pointing object 56 0 (offset 0)
Ignoring wrong pointing object 58 0 (offset 0)
Ignoring wrong pointing object 60 0 (offset 0)
Ignoring wrong pointing object 62 0 (offset 0)
Ignoring wrong 

In [17]:
from pprint import pprint
import time
import json


descriptions = []
descriptions_per_point = {}
responses = {}

all_references = []

for i in range(initial_points_df.shape[0]):
    if os.path.exists(f"./generated_descriptions_Phi-4-mini-instruct/{i}_response.json"):
        continue
    file = initial_points_df.iloc[i].File
        
    data_group = initial_points_df.iloc[i].Data_groups
    data_point = initial_points_df.iloc[i].Data_points

    prompt = f"What is the meaning of {data_point} in relation to {data_group}?"
    
    # query_engine = documents[file].as_query_engine()
    response = query_engine.query(prompt)

    print(f'index: {i}')
    
    point_i_response = {}

    point_i_response["question"] =  prompt
    point_i_response["original_source"] =  file
    point_i_response["data_group"] =  data_group
    point_i_response["data_point"] =  data_point


    references = []
    for node_with_score in response.source_nodes:
        node = node_with_score.node  # Access the underlying Node
        reference = {}
        reference["text"] = node.get_text() # Node text 
        reference["metadata"] = node.metadata  # Metadata if any
        reference["score"] = node_with_score.score # Similarity score
        references.append(reference)

    point_i_response["reference_1"] =  references[0]["metadata"]["file_name"]
    point_i_response["reference_2"] =  references[1]["metadata"]["file_name"]
    point_i_response["description"] =  response.response
    point_i_response["references"] = references

    with open(f"./generated_descriptions_Phi-4-mini-instruct/{i}_response.json", "w") as f:
        json.dump(point_i_response, f)
    
    descriptions.append(response.response)
    descriptions_per_point[data_point] = response.response
    responses[i] = response
    initial_points_df.loc[i, 'description'] = response.response
    
    all_references.append(point_i_response["reference_1"])
    all_references.append(point_i_response["reference_2"])

    
    


index: 1
index: 2
index: 3
index: 4
index: 5
index: 6
index: 7
index: 8
index: 9
index: 10
index: 11
index: 12
index: 13
index: 14
index: 15
index: 16
index: 17
index: 18
index: 19
index: 20
index: 21
index: 22
index: 23
index: 24
index: 25
index: 26
index: 27
index: 28
index: 29
index: 30
index: 31
index: 32
index: 33
index: 34
index: 35
index: 36
index: 37
index: 38
index: 39
index: 40
index: 41
index: 42
index: 43
index: 44
index: 45
index: 46
index: 47
index: 48
index: 49
index: 50
index: 51
index: 52
index: 53
index: 54
index: 55
index: 56
index: 57
index: 58
index: 59
index: 60
index: 61
index: 62
index: 63
index: 64
index: 65
index: 66
index: 67
index: 68
index: 69
index: 70
index: 71
index: 72
index: 73
index: 74
index: 75
index: 76
index: 77
index: 78
index: 79
index: 80
index: 81
index: 82
index: 83
index: 84
index: 85
index: 86
index: 87
index: 88
index: 89
index: 90
index: 91
index: 92
index: 93
index: 94
index: 95
index: 96
index: 97
index: 98
index: 99
index: 100
index: 1

In [13]:
Counter(all_references)

Counter({'CPR 2024.pdf': 2})

In [14]:
len(Counter(all_references).keys())

for file in files:
    if file not in Counter(all_references).keys():
        print (file)

Munaro and Tavares 2021.pdf
Stratmann 2023.pdf
Göswein 2022.pdf
Atta 2021.pdf
Bauen digital Schweiz 2024.pdf
Bosma 2024.pdf
Circularise 2025.pdf
Byers 2025.pdf
Çetin 2023.pdf
Giovanardi 2023.pdf
Heisel and Rau-Oberhuber 2020.pdf
Jensen 2023.pdf
Honic 2021.pdf
ISO 59040 2025.pdf
KC 2024.pdf
Honic 2019.pdf
Lopes and Barata 2024.pdf
Kebede 2024.pdf
Seddiqui 2024.pdf
Mulhall 2022.pdf
Mao and Cao 2025.pdf
Markou 2025.pdf
Platform CB 2023.pdf
Wan and Jiang 2025.pdf
Van Capelleveen 2023.pdf
Christensen 2025.pdf
BAMB 2019.pdf
Ruismäki 2025.pdf
ESPR 2024.pdf
